In [1]:
import cv2, os, pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import GradientBoostingClassifier, AdaBoostClassifier, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from skimage.feature import local_binary_pattern, graycomatrix, graycoprops
from scipy.stats import skew, kurtosis
import xgboost as xgb

In [2]:
# ===== LOAD IMAGES =====
def load_images_from_folder(folder, label):
    images, labels = [], []
    for filename in os.listdir(folder):
        img = cv2.imread(os.path.join(folder,filename))
        if img is not None:
            images.append(img)
            labels.append(label)
    return images, labels

berkuah_images, berkuah_labels = load_images_from_folder('Makanan/Berkuah', 0)
tidak_berkuah_images, tidak_berkuah_labels = load_images_from_folder('Makanan/Tidak berkuah', 1)
images = berkuah_images + tidak_berkuah_images
labels = berkuah_labels + tidak_berkuah_labels

In [3]:
# ===== FEATURE EXTRACTORS =====
def extract_lbp_features(image):
    resized = cv2.resize(image, (128,128))
    gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)
    lbp = local_binary_pattern(gray, 24, 3, method='uniform')
    hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, 26), range=(0, 25))
    hist = hist.astype("float") / (hist.sum() + 1e-7)
    return hist

def extract_color_features(image):
    resized = cv2.resize(image, (128,128))
    hsv = cv2.cvtColor(resized, cv2.COLOR_BGR2HSV)
    h,s,v = cv2.split(hsv)
    feats = [
        np.mean(h), np.std(h), skew(h.flatten()), kurtosis(h.flatten()),
        np.mean(s), np.std(s), skew(s.flatten()), kurtosis(s.flatten()),
        np.mean(v), np.std(v), skew(v.flatten()), kurtosis(v.flatten())
    ]
    return np.array(feats)

def extract_haralick_features_simple(image):
    resized = cv2.resize(image, (128,128))
    gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)
    glcm = graycomatrix(gray, [5], [0], levels=256, symmetric=True, normed=True)
    props = ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'ASM']
    feats = [graycoprops(glcm, p)[0,0] for p in props]
    return np.array(feats)

def extract_combined_features(image):
    return np.hstack([
        extract_lbp_features(image),
        extract_color_features(image),
        extract_haralick_features_simple(image)
    ])

feature_extractors = {
    "Combined": extract_combined_features
}

In [ ]:

# ===== TRAINING LOOP =====
results = []
best_acc = -1
best_model = None
best_extractor_name = None
scaler = StandardScaler()

for feat_name, extractor in feature_extractors.items():
    print(f"\n=== Menggunakan fitur: {feat_name} ===")
    X = np.array([extractor(img) for img in images])
    X = scaler.fit_transform(X)
    export_data = {
    "model": best_model,
    "scaler": scaler,          # pastikan ini ada
    "feature_extractor": best_extractor_name
    }

    with open("model_makanan_full.pkl", "wb") as f:
        pickle.dump(export_data, f)

    y = np.array(labels)
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    classifiers = {
        "RandomForest": RandomForestClassifier(random_state=42),
        "GradientBoosting": GradientBoostingClassifier(random_state=42),
        "AdaBoost": AdaBoostClassifier(random_state=42),
        "KNN": KNeighborsClassifier(),
        "XGBoost": xgb.XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
    }
    
    param_grids = {
        "RandomForest": {'n_estimators':[100], 'max_depth':[None]},
        "GradientBoosting": {'n_estimators':[100], 'learning_rate':[0.1], 'max_depth':[3]},
        "AdaBoost": {'n_estimators':[100], 'learning_rate':[1.0]},
        "KNN": {'n_neighbors':[5]},
        "XGBoost": {'n_estimators':[100], 'learning_rate':[0.1], 'max_depth':[3]}
    }
    
    for clf_name, clf in classifiers.items():
        grid = GridSearchCV(clf, param_grids[clf_name], cv=3, scoring='accuracy', n_jobs=-1)
        grid.fit(X_train, y_train)
        y_pred = grid.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        report = classification_report(y_test, y_pred, target_names=['Berkuah','Tidak Berkuah'], output_dict=True)
        
        results.append({
            "Fitur": feat_name,
            "Model": clf_name,
            "Akurasi": round(acc,3),
            "Presisi_Berkuah": round(report['Berkuah']['precision'],3),
            "Recall_Berkuah": round(report['Berkuah']['recall'],3),
            "F1_Berkuah": round(report['Berkuah']['f1-score'],3),
            "Presisi_TdkBerkuah": round(report['Tidak Berkuah']['precision'],3),
            "Recall_TdkBerkuah": round(report['Tidak Berkuah']['recall'],3),
            "F1_TdkBerkuah": round(report['Tidak Berkuah']['f1-score'],3)
        })
        
        if acc > best_acc:
            best_acc = acc
            best_model = grid.best_estimator_
            best_extractor_name = feat_name


=== Menggunakan fitur: Combined ===


C:\Users\Vania Oriana Tanoto\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\xgboost\training.py:183: UserWarning: [00:21:25] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [5]:
# Simpan hasil rapi
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="Akurasi", ascending=False)
print("\n=== Ringkasan Hasil ===")
print(results_df.to_string(index=False))

# Simpan model + scaler + extractor
export_data = {
    "model": best_model,
    "scaler": scaler,
    "feature_extractor": best_extractor_name
}
with open("model_makanan_full.pkl", "wb") as f:
    pickle.dump(export_data, f)
print(f"\n✔ Model terbaik disimpan dengan fitur: {best_extractor_name}, akurasi: {best_acc:.3f}")


=== Ringkasan Hasil ===
   Fitur            Model  Akurasi  Presisi_Berkuah  Recall_Berkuah  F1_Berkuah  Presisi_TdkBerkuah  Recall_TdkBerkuah  F1_TdkBerkuah
Combined         AdaBoost    0.444              0.5             0.4       0.444                0.40               0.50          0.444
Combined          XGBoost    0.444              0.5             0.4       0.444                0.40               0.50          0.444
Combined              KNN    0.444              0.5             0.4       0.444                0.40               0.50          0.444
Combined     RandomForest    0.333              0.4             0.4       0.400                0.25               0.25          0.250
Combined GradientBoosting    0.333              0.4             0.4       0.400                0.25               0.25          0.250

✔ Model terbaik disimpan dengan fitur: Combined, akurasi: 0.444


NameError: name 'image_np' is not defined